In [1]:
from __future__ import annotations
import enum


In [2]:
import jax
import jax.numpy as jnp
jax.config.update('jax_platform_name', 'cpu') 
jax.config.update("jax_num_cpu_devices", 8)

In [3]:
default_device = jax.devices()[0]
print(default_device)

cpu:0


In [4]:
jax.set_mesh(jax.make_mesh((4, 2), ("X", "Y")))

x = jnp.arange(8 * 4.).reshape(8, 4)
x = jax.device_put(x, jax.P("X", "Y"))
print(jax.typeof(x))

float32[8@X,4@Y]


In [5]:
jax.debug.visualize_array_sharding(x)

                  
  CPU 0    CPU 1  
                  
                  
  CPU 2    CPU 3  
                  
                  
  CPU 4    CPU 5  
                  
                  
  CPU 6    CPU 7  
                  

In [6]:
y = jnp.sin(x).T
print(jax.typeof(y))

float32[4@Y,8@X]


In [7]:
print(jax.sharding.get_mesh())

Mesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit))


In [8]:
@jax.jit
def f(x):
    abstract_mesh = jax.sharding.AbstractMesh((8,), ("A",), (jax.sharding.AxisType.Explicit,))
    with jax.sharding.use_abstract_mesh(abstract_mesh):
        y = jax.reshard(x, jax.P("A", None))
        return y * 2

z = f(x)
print(jax.typeof(z))

float32[8@A,4]


In [9]:
print(x.sharding)

NamedSharding(mesh=Mesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', 'Y'), memory_kind=device)


In [10]:
jax.debug.visualize_array_sharding(x)

                  
  CPU 0    CPU 1  
                  
                  
  CPU 2    CPU 3  
                  
                  
  CPU 4    CPU 5  
                  
                  
  CPU 6    CPU 7  
                  

In [11]:
for s in x.addressable_shards:
    print(s.device, s.data, sep="\n", end="\n\n")

cpu:0
[[0. 1.]
 [4. 5.]]

cpu:1
[[2. 3.]
 [6. 7.]]

cpu:2
[[ 8.  9.]
 [12. 13.]]

cpu:3
[[10. 11.]
 [14. 15.]]

cpu:4
[[16. 17.]
 [20. 21.]]

cpu:5
[[18. 19.]
 [22. 23.]]

cpu:6
[[24. 25.]
 [28. 29.]]

cpu:7
[[26. 27.]
 [30. 31.]]



In [12]:
x

Array([[ 0.,  1.,  2.,  3.],
       [ 4.,  5.,  6.,  7.],
       [ 8.,  9., 10., 11.],
       [12., 13., 14., 15.],
       [16., 17., 18., 19.],
       [20., 21., 22., 23.],
       [24., 25., 26., 27.],
       [28., 29., 30., 31.]], dtype=float32)

In [13]:
y = jax.device_put(x, jax.P("Y", "X"))
print(y.sharding)
jax.debug.visualize_array_sharding(y)

NamedSharding(mesh=Mesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('Y', 'X'), memory_kind=device)


                                    
                                    
  CPU 0    CPU 2    CPU 4    CPU 6  
                                    
                                    
                                    
                                    
                                    
  CPU 1    CPU 3    CPU 5    CPU 7  
                                    
                                    
                                    

In [14]:
y = jax.device_put(x, jax.P("X", None))
print(y.sharding)
jax.debug.visualize_array_sharding(y)

NamedSharding(mesh=Mesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', None), memory_kind=device)


            
  CPU 0,1   
            
            
  CPU 2,3   
            
            
  CPU 4,5   
            
            
  CPU 6,7   
            

In [15]:
for s in y.addressable_shards:
    print(s.device, s.data, sep="\n", end="\n\n")

cpu:0
[[0. 1. 2. 3.]
 [4. 5. 6. 7.]]

cpu:1
[[0. 1. 2. 3.]
 [4. 5. 6. 7.]]

cpu:2
[[ 8.  9. 10. 11.]
 [12. 13. 14. 15.]]

cpu:3
[[ 8.  9. 10. 11.]
 [12. 13. 14. 15.]]

cpu:4
[[16. 17. 18. 19.]
 [20. 21. 22. 23.]]

cpu:5
[[16. 17. 18. 19.]
 [20. 21. 22. 23.]]

cpu:6
[[24. 25. 26. 27.]
 [28. 29. 30. 31.]]

cpu:7
[[24. 25. 26. 27.]
 [28. 29. 30. 31.]]



In [16]:
y = jax.device_put(x, jax.P(("X", "Y")))
print(y.sharding)
jax.debug.visualize_array_sharding(y)

NamedSharding(mesh=Mesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P(('X', 'Y'),), memory_kind=device)


   CPU 0    
            
   CPU 1    
            
   CPU 2    
            
   CPU 3    
            
   CPU 4    
            
   CPU 5    
            
   CPU 6    
            
   CPU 7    
            

In [17]:
for s in y.addressable_shards:
    print(s.device, s.data, sep="\n", end="\n\n")

cpu:0
[[0. 1. 2. 3.]]

cpu:1
[[4. 5. 6. 7.]]

cpu:2
[[ 8.  9. 10. 11.]]

cpu:3
[[12. 13. 14. 15.]]

cpu:4
[[16. 17. 18. 19.]]

cpu:5
[[20. 21. 22. 23.]]

cpu:6
[[24. 25. 26. 27.]]

cpu:7
[[28. 29. 30. 31.]]



In [18]:
y = jax.device_put(x, jax.P("X", None, unreduced={"Y"}))
print(y.sharding)

NamedSharding(mesh=Mesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', None, unreduced={'Y'}), memory_kind=device)


In [19]:
for s in y.addressable_shards:
    print(s.device, s.data, sep="\n", end="\n\n")

cpu:0
[[0. 1. 0. 0.]
 [4. 5. 0. 0.]]

cpu:1
[[0. 0. 2. 3.]
 [0. 0. 6. 7.]]

cpu:2
[[ 8.  9.  0.  0.]
 [12. 13.  0.  0.]]

cpu:3
[[ 0.  0. 10. 11.]
 [ 0.  0. 14. 15.]]

cpu:4
[[16. 17.  0.  0.]
 [20. 21.  0.  0.]]

cpu:5
[[ 0.  0. 18. 19.]
 [ 0.  0. 22. 23.]]

cpu:6
[[24. 25.  0.  0.]
 [28. 29.  0.  0.]]

cpu:7
[[ 0.  0. 26. 27.]
 [ 0.  0. 30. 31.]]



In [20]:
x

Array([[ 0.,  1.,  2.,  3.],
       [ 4.,  5.,  6.,  7.],
       [ 8.,  9., 10., 11.],
       [12., 13., 14., 15.],
       [16., 17., 18., 19.],
       [20., 21., 22., 23.],
       [24., 25., 26., 27.],
       [28., 29., 30., 31.]], dtype=float32)

In [21]:
mesh2 = jax.make_mesh((8,), ("A",))
z = jax.device_put(x, jax.NamedSharding(mesh2, jax.P("A", None)))
print(z.sharding)
print(x.sharding)

NamedSharding(mesh=Mesh('A': 8, axis_types=(Explicit,)), spec=P('A', None), memory_kind=device)
NamedSharding(mesh=Mesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', 'Y'), memory_kind=device)


In [22]:
print(jax.typeof(x).sharding)

NamedSharding(mesh=AbstractMesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit), device_kind=cpu, num_cores=None), spec=P('X', 'Y'))


In [23]:
jax.jit(lambda x: print(jax.typeof(x).sharding))(x)

NamedSharding(mesh=AbstractMesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit), device_kind=cpu, num_cores=None), spec=P('X', 'Y'))


In [24]:
jax.typeof(x)

ShapedArray(float32[8@X,4@Y])

In [25]:
jax.typeof(x).sharding.mesh

AbstractMesh('X': 4, 'Y': 2, axis_types=(Explicit, Explicit), device_kind=cpu, num_cores=None)

In [26]:
arg0 = jax.device_put(jnp.arange(4).reshape(4, 1), jax.P("X", None))

In [27]:
arg0

Array([[0],
       [1],
       [2],
       [3]], dtype=int32)

In [28]:
for s in arg0.addressable_shards:
    print(s.device, s.data, sep="\n", end="\n\n")

jax.debug.visualize_array_sharding(arg0)

cpu:0
[[0]]

cpu:1
[[0]]

cpu:2
[[1]]

cpu:3
[[1]]

cpu:4
[[2]]

cpu:5
[[2]]

cpu:6
[[3]]

cpu:7
[[3]]



         
 CPU 0,1 
         
         
 CPU 2,3 
         
         
 CPU 4,5 
         
         
 CPU 6,7 
         

In [29]:
arg1 = jax.device_put(jnp.arange(8).reshape(1, 8), jax.P(None, "Y"))

In [30]:
for s in arg1.addressable_shards:
    print(s.device, s.data, sep="\n", end="\n\n")

cpu:0
[[0 1 2 3]]

cpu:1
[[4 5 6 7]]

cpu:2
[[0 1 2 3]]

cpu:3
[[4 5 6 7]]

cpu:4
[[0 1 2 3]]

cpu:5
[[4 5 6 7]]

cpu:6
[[0 1 2 3]]

cpu:7
[[4 5 6 7]]



In [31]:
jax.debug.visualize_array_sharding(arg1)

In [32]:
result = arg0 + arg1

In [33]:
result

Array([[ 0,  1,  2,  3,  4,  5,  6,  7],
       [ 1,  2,  3,  4,  5,  6,  7,  8],
       [ 2,  3,  4,  5,  6,  7,  8,  9],
       [ 3,  4,  5,  6,  7,  8,  9, 10]], dtype=int32)

In [34]:
arg0

Array([[0],
       [1],
       [2],
       [3]], dtype=int32)

In [35]:
arg1

Array([[0, 1, 2, 3, 4, 5, 6, 7]], dtype=int32)

In [36]:
print(f"{jax.typeof(arg0)=!s}")
print(f"{jax.typeof(arg1)=!s}")
print(f"{jax.typeof(result)=!s}")

jax.typeof(arg0)=int32[4@X,1]
jax.typeof(arg1)=int32[1,8@Y]
jax.typeof(result)=int32[4@X,8@Y]


In [37]:
@jax.jit
def add_arrays(x, y):
  ans = x + y
  print(f"{jax.typeof(arg0)=!s}")
  print(f"{jax.typeof(arg1)=!s}")
  print(f"{jax.typeof(result)=!s}")
  return ans

add_arrays(arg0, arg1)

jax.typeof(arg0)=int32[4@X,1]
jax.typeof(arg1)=int32[1,8@Y]
jax.typeof(result)=int32[4@X,8@Y]


Array([[ 0,  1,  2,  3,  4,  5,  6,  7],
       [ 1,  2,  3,  4,  5,  6,  7,  8],
       [ 2,  3,  4,  5,  6,  7,  8,  9],
       [ 3,  4,  5,  6,  7,  8,  9, 10]], dtype=int32)

In [38]:
compile_txt = jax.jit(lambda x: x.sum(0)).lower(x).compile().as_text()
print('all-reduce(' in compile_txt)

True


In [39]:
Auto = jax.sharding.AxisType.Auto
auto_mesh = jax.make_mesh((4, 2), ('X', 'Y'), (Auto, Auto))
jax.set_mesh(auto_mesh)

x = jax.device_put(jnp.arange(8 * 4. ).reshape(8, 4 ), jax.P(None, 'X'))
y = jax.device_put(jnp.arange(4 * 16.).reshape(4, 16), jax.P('X', None))

z = jnp.dot(x, y)  # not an error!

In [40]:
z

Array([[ 224.,  230.,  236.,  242.,  248.,  254.,  260.,  266.,  272.,
         278.,  284.,  290.,  296.,  302.,  308.,  314.],
       [ 608.,  630.,  652.,  674.,  696.,  718.,  740.,  762.,  784.,
         806.,  828.,  850.,  872.,  894.,  916.,  938.],
       [ 992., 1030., 1068., 1106., 1144., 1182., 1220., 1258., 1296.,
        1334., 1372., 1410., 1448., 1486., 1524., 1562.],
       [1376., 1430., 1484., 1538., 1592., 1646., 1700., 1754., 1808.,
        1862., 1916., 1970., 2024., 2078., 2132., 2186.],
       [1760., 1830., 1900., 1970., 2040., 2110., 2180., 2250., 2320.,
        2390., 2460., 2530., 2600., 2670., 2740., 2810.],
       [2144., 2230., 2316., 2402., 2488., 2574., 2660., 2746., 2832.,
        2918., 3004., 3090., 3176., 3262., 3348., 3434.],
       [2528., 2630., 2732., 2834., 2936., 3038., 3140., 3242., 3344.,
        3446., 3548., 3650., 3752., 3854., 3956., 4058.],
       [2912., 3030., 3148., 3266., 3384., 3502., 3620., 3738., 3856.,
        3974., 4092., 4210

In [41]:
for s in z.addressable_shards:
    print(s.device, s.data, sep="\n", end="\n\n")

cpu:0
[[ 224.  230.  236.  242.  248.  254.  260.  266.  272.  278.  284.  290.
   296.  302.  308.  314.]
 [ 608.  630.  652.  674.  696.  718.  740.  762.  784.  806.  828.  850.
   872.  894.  916.  938.]
 [ 992. 1030. 1068. 1106. 1144. 1182. 1220. 1258. 1296. 1334. 1372. 1410.
  1448. 1486. 1524. 1562.]
 [1376. 1430. 1484. 1538. 1592. 1646. 1700. 1754. 1808. 1862. 1916. 1970.
  2024. 2078. 2132. 2186.]
 [1760. 1830. 1900. 1970. 2040. 2110. 2180. 2250. 2320. 2390. 2460. 2530.
  2600. 2670. 2740. 2810.]
 [2144. 2230. 2316. 2402. 2488. 2574. 2660. 2746. 2832. 2918. 3004. 3090.
  3176. 3262. 3348. 3434.]
 [2528. 2630. 2732. 2834. 2936. 3038. 3140. 3242. 3344. 3446. 3548. 3650.
  3752. 3854. 3956. 4058.]
 [2912. 3030. 3148. 3266. 3384. 3502. 3620. 3738. 3856. 3974. 4092. 4210.
  4328. 4446. 4564. 4682.]]

cpu:1
[[ 224.  230.  236.  242.  248.  254.  260.  266.  272.  278.  284.  290.
   296.  302.  308.  314.]
 [ 608.  630.  652.  674.  696.  718.  740.  762.  784.  806.  828.  850.
   

In [42]:
for s in x.addressable_shards:
    print(s.device, s.data, sep="\n", end="\n\n")

cpu:0
[[ 0.]
 [ 4.]
 [ 8.]
 [12.]
 [16.]
 [20.]
 [24.]
 [28.]]

cpu:1
[[ 0.]
 [ 4.]
 [ 8.]
 [12.]
 [16.]
 [20.]
 [24.]
 [28.]]

cpu:2
[[ 1.]
 [ 5.]
 [ 9.]
 [13.]
 [17.]
 [21.]
 [25.]
 [29.]]

cpu:3
[[ 1.]
 [ 5.]
 [ 9.]
 [13.]
 [17.]
 [21.]
 [25.]
 [29.]]

cpu:4
[[ 2.]
 [ 6.]
 [10.]
 [14.]
 [18.]
 [22.]
 [26.]
 [30.]]

cpu:5
[[ 2.]
 [ 6.]
 [10.]
 [14.]
 [18.]
 [22.]
 [26.]
 [30.]]

cpu:6
[[ 3.]
 [ 7.]
 [11.]
 [15.]
 [19.]
 [23.]
 [27.]
 [31.]]

cpu:7
[[ 3.]
 [ 7.]
 [11.]
 [15.]
 [19.]
 [23.]
 [27.]
 [31.]]

